# MT Evaluation Workflow — Template

Template for evaluating machine translation models. Covers:
1. Dataset loading
2. Translation (mbart many-to-many, plus patterns for other models)
3. METEOR evaluation
4. BERTScore evaluation
5. Error analysis

See [WORKFLOW.md](../WORKFLOW.md) for full methodology. See [evaluation-template.md](../evaluation-template.md) for the documentation template.

**Google Colab:** run the setup cell below.  
**Local:** `pip install transformers sentencepiece bert_score sacrebleu nltk`

## Setup

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score sacrebleu nltk accelerate

import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from bert_score import score as bert_score
import nltk

## Load dataset

Replace the dummy data below with your parallel corpus.  
Required columns: `source` (original language text), `target` (gold standard English translation).  
Dataset must not have been used in training the candidate models.

In [ ]:
# Replace with your parallel corpus
# Required columns: 'source' (original language), 'target' (gold standard English)
# Example: Tatoeba test data from https://github.com/Helsinki-NLP/Tatoeba-Challenge
df = pd.DataFrame({
    'source': ['Replace with source language sentence 1.',
               'Replace with source language sentence 2.',
               'Replace with source language sentence 3.'],
    'target': ['Replace with gold standard English 1.',
               'Replace with gold standard English 2.',
               'Replace with gold standard English 3.'],
})
print(f'Dataset: {len(df)} rows')
df.head()

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.data[idx], return_tensors="pt",
            padding="max_length", truncation=True, max_length=self.max_length
        )
        return {k: v.squeeze() for k, v in encoded.items()}

## Translate

Default model: `facebook/mbart-large-50-many-to-many-mmt` (many-to-many).  
Change `SRC_LANG` to your source language code — see the [model card](https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt) for all supported codes.  
For other model types (Helsinki-NLP one-to-one, madlad400, etc.) see commented examples below.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
SRC_LANG   = "xx_XX"   # <<<< replace with your language code (e.g. "es_XX", "ar_AR", "tr_TR", "af_ZA")
TGT_LANG   = "en_XX"
BATCH_SIZE = 16

# ── Load model ────────────────────────────────────────────────────────────────
model     = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Running on: {device}')

# ── Translate ─────────────────────────────────────────────────────────────────
dataset    = TranslationDataset(df['source'].tolist(), tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
translations = []

for batch in tqdm(dataloader):
    data = {k: v.to(device) for k, v in batch.items()}
    tokens = model.generate(**data, forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG])
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))

df['translation'] = translations
df.head()

## Evaluate — METEOR

In [ ]:
meteor = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(df['target'], df['translation'])):
    score = meteor([ref.split()], hyp.split())
    scores.append(score)

df['meteor'] = scores
print(f"Average METEOR: {np.mean(scores):.4f}")
df[['source','target','translation','meteor']].head(10)

## Evaluate — BERTScore

In [ ]:
P, R, F1 = bert_score(df['translation'].tolist(), df['target'].tolist(), lang='en', verbose=True)
df['bertscore'] = F1.numpy()
print(f"Average BERTScore F1: {F1.mean():.4f}")

## Error analysis

Sample low-scoring translations and inspect manually. Look for:
- Named entity preservation
- Idiomatic language failures
- Tokenisation / truncation issues
- Code switching
- Domain mismatch patterns

In [ ]:
# Summary statistics
summary = pd.DataFrame({
    'bertscore': df['bertscore'].describe(),
    'meteor':    df['meteor'].describe()
})
print(summary)

# Sample low-scoring translations for manual review
low = df[df['bertscore'] < df['bertscore'].quantile(0.25)].sample(min(10, len(df)))
for _, row in low.iterrows():
    print(f"\nSOURCE:      {row['source']}")
    print(f"GOLD:        {row['target']}")
    print(f"TRANSLATION: {row['translation']}")
    print(f"METEOR: {row['meteor']:.3f}  BERTScore: {row['bertscore']:.3f}")